# Notebook 11 — Speech-to-Speech (GPT-4o Realtime)

In nb08 we built the **sequential** voice agent: STT → LLM → TTS, three separate models, every stage logged and debuggable. This notebook builds the same agent the **other** way — with **GPT-4o Realtime**, a single audio-in / audio-out model — and measures the gap that motivates choosing one over the other (S3 §6.4).

## Why care?

Sequential pipelines lose ~all of the speech's *prosody* the moment STT collapses audio into text. The user said *"What's the WEATHER in Tokyo?!"* with stress on the W word and frustrated rising tone — the LLM gets `"What's the weather in Tokyo?"`. Any answer the LLM produces and TTS speaks will be flat, no matter how good the acting in the LLM's word choice.

Speech-to-speech models keep the audio modality end-to-end. They can pause, interject, sound surprised, slow down for emphasis. They also cut latency roughly in half because there are no STT or TTS round-trips — the model directly emits audio tokens.

**The cost:** they're harder to debug (audio-in / audio-out is a black box), tool-calling is less mature, and you pay premium per-minute audio pricing. The decision is not abstract — it's a per-product call.

## What this notebook does

1. Convert nb08's TTS-generated input queries to PCM16 (Realtime's expected format).
2. Open a WebSocket session with `gpt-4o-realtime-preview`.
3. Stream the input audio in, collect the output audio chunks, measure time-to-first-audio.
4. Compare numbers + listen to the difference vs nb08.

## 1. Prepare PCM16 input

OpenAI Realtime expects **24 kHz mono PCM16**. We convert nb08's MP3 inputs via ffmpeg.

In [ ]:
import asyncio, base64, io, os, subprocess, time, wave
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv(dotenv_path="../.env")
client = AsyncOpenAI()

AUDIO_DIR = Path("../data/audio_samples/agent")
OUT_DIR   = Path("../data/audio_samples/realtime")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def mp3_to_pcm16_24k(mp3_path: Path) -> bytes:
    """Decode + resample to mono 24kHz, return raw little-endian s16 bytes."""
    out = subprocess.run(
        ["ffmpeg", "-i", str(mp3_path), "-loglevel", "error",
         "-ar", "24000", "-ac", "1", "-f", "s16le", "pipe:1"],
        capture_output=True, check=True,
    )
    return out.stdout

INPUTS = {
    "en":     AUDIO_DIR / "in_en.mp3",
    "zh":     AUDIO_DIR / "in_zh.mp3",
    "mixed":  AUDIO_DIR / "in_mixed.mp3",
}
for tag, p in INPUTS.items():
    assert p.exists(), f"run nb08 first to generate {p}"
    print(f"{tag}: {p.name}  ({p.stat().st_size / 1024:.0f} KB mp3)")

## 2. The Realtime turn — async helper

Pattern (per OpenAI's Realtime docs):

1. `connect()` → opens a WebSocket session.
2. `session.update` → set output modalities, voice, audio formats, system prompt.
3. `input_audio_buffer.append` → stream the user's audio.
4. `input_audio_buffer.commit` + `response.create` → tell the model to start replying.
5. Iterate over events. Audio comes in as `response.audio.delta` (base64 PCM16 chunks). Done event closes the turn.

In [ ]:
REALTIME_MODEL = "gpt-4o-realtime-preview"   # update if a newer pinned variant ships

INSTRUCTIONS = (
    "You are a concise voice assistant. Answer in one or two short sentences. "
    "Match the user's language (English, Chinese, or mixed)."
)

def write_pcm16_wav(out_path: Path, pcm: bytes, sample_rate: int = 24000):
    with wave.open(str(out_path), "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sample_rate)
        w.writeframes(pcm)

async def realtime_turn(input_pcm: bytes, save_to: Path) -> dict:
    """Send one audio query, collect the audio response. Returns timing dict."""
    t_start = time.time()
    t_first_audio = None
    out_pcm = bytearray()
    transcript = ""

    async with client.beta.realtime.connect(model=REALTIME_MODEL) as conn:
        await conn.session.update(session={
            "modalities": ["audio", "text"],
            "instructions": INSTRUCTIONS,
            "voice": "alloy",
            "input_audio_format":  "pcm16",
            "output_audio_format": "pcm16",
            "turn_detection": None,   # we drive the turn manually
        })
        await conn.input_audio_buffer.append(audio=base64.b64encode(input_pcm).decode())
        await conn.input_audio_buffer.commit()
        await conn.response.create()

        async for event in conn:
            if event.type == "response.audio.delta":
                if t_first_audio is None:
                    t_first_audio = time.time() - t_start
                out_pcm.extend(base64.b64decode(event.delta))
            elif event.type == "response.audio_transcript.delta":
                transcript += event.delta
            elif event.type == "response.done":
                break
            elif event.type == "error":
                raise RuntimeError(f"Realtime error: {event}")

    write_pcm16_wav(save_to, bytes(out_pcm))
    return {
        "time_to_first_audio": t_first_audio,
        "total_time":          time.time() - t_start,
        "output_seconds":      len(out_pcm) / (24000 * 2),
        "transcript":          transcript.strip(),
    }

## 3. Run the three queries

In [ ]:
from IPython.display import Audio, display

results = {}
for tag, in_path in INPUTS.items():
    pcm = mp3_to_pcm16_24k(in_path)
    out_path = OUT_DIR / f"out_realtime_{tag}.wav"
    metrics = await realtime_turn(pcm, out_path)
    results[tag] = metrics
    print(f"\n=== {tag} ===")
    print(f"  transcript:           {metrics['transcript']}")
    print(f"  time-to-first-audio:  {metrics['time_to_first_audio']*1000:.0f} ms")
    print(f"  total response time:  {metrics['total_time']*1000:.0f} ms")
    print(f"  spoken length:        {metrics['output_seconds']:.1f} s")
    display(Audio(str(out_path)))

## 4. Compare — sequential (nb08) vs streaming (nb08) vs realtime (nb11)

We'll re-run the sync and streaming nb08 versions for the same queries, then put all three side by side. (If you ran nb08 in the same session and have `sync_results`, you can skip the re-run cell.)

In [ ]:
# Re-import nb08's functions if not already in scope. Simplest: copy the
# minimal sync pipeline here so this notebook stands alone.
from faster_whisper import WhisperModel
asr = WhisperModel("large-v3", device="cpu", compute_type="int8")
openai_sync = client.with_options()  # AsyncOpenAI is fine; we'll await it directly

async def nb08_sync_total(in_path: Path) -> float:
    """Re-run nb08's synchronous pipeline (no tool, just answer) and return total seconds."""
    t0 = time.time()
    segments, _ = asr.transcribe(str(in_path), vad_filter=True)
    user_text = " ".join(s.text.strip() for s in segments)
    r = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user",   "content": user_text},
        ],
    )
    answer = r.choices[0].message.content
    audio = await client.audio.speech.create(model="tts-1", voice="alloy", input=answer)
    audio.stream_to_file(str(OUT_DIR / f"sync_{in_path.stem}.mp3"))
    return time.time() - t0

sync_totals = {tag: await nb08_sync_total(p) for tag, p in INPUTS.items()}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tags = list(INPUTS)
sync_total_ms     = [sync_totals[t] * 1000 for t in tags]
realtime_first_ms = [results[t]["time_to_first_audio"] * 1000 for t in tags]
realtime_total_ms = [results[t]["total_time"] * 1000 for t in tags]

x = np.arange(len(tags)); w = 0.27
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w, sync_total_ms,     width=w, label="nb08 sync (total)")
ax.bar(x,     realtime_first_ms, width=w, label="nb11 realtime (first audio)")
ax.bar(x + w, realtime_total_ms, width=w, label="nb11 realtime (full reply)")
ax.axhline(800, color="red", linestyle="--", label="800 ms — broken-feeling")
ax.set_xticks(x); ax.set_xticklabels(tags)
ax.set_ylabel("ms")
ax.set_title("Sequential pipeline vs Realtime — same queries")
ax.legend()
plt.tight_layout(); plt.show()

for t in tags:
    print(f"{t:>6}  sync_total={sync_totals[t]*1000:>5.0f} ms   realtime_first={results[t]['time_to_first_audio']*1000:>5.0f} ms")

## 5. Listen to both — the prosody difference

Numbers tell only part of the story. **Play the sync output then the realtime output for the same query**. The realtime version typically has natural pauses, emphasis on the city name, falling tone at end of statement. The sync version reads the answer flatly because TTS synthesizes from text alone and has no signal about how the question was asked.

In [ ]:
for tag in tags:
    print(f"\n=== {tag} ===")
    print("  nb08 sync (sequential STT→LLM→TTS):")
    display(Audio(str(OUT_DIR / f"sync_in_{tag}.mp3")))
    print("  nb11 realtime (speech-to-speech):")
    display(Audio(str(OUT_DIR / f"out_realtime_{tag}.wav")))

## 6. The decision rule

From S3 §6.4, refined by what you just measured:

| Concern | Sequential (nb08) | Realtime (nb11) |
|---|---|---|
| Latency to first audio | 600–1200 ms (streaming) | **300–500 ms** ✓ |
| Voice expressiveness | Limited (text-shaped TTS) | **High** ✓ |
| Tool calling maturity | **Mature** ✓ | Still maturing |
| Per-turn cost | Cheap (~$0.001) | **5–20× higher** |
| Debuggability | **High** — log every text step ✓ | Lower — audio is opaque |
| Provider portability | Mix vendors freely ✓ | Locked to OpenAI |
| Multilingual code-switching | Whisper handles it (with strategies from nb07) ✓ | Excellent natively ✓ |

**Pick by what matters for your product:**
- Customer support bot with database tools, audit logging, FinOps pressure → **sequential**.
- Language-learning conversation partner, accessibility companion, mood-aware coach → **realtime**.
- Hybrid is rare but real: sequential as the default with a feature flag to escalate to realtime when the user's session is detected as emotional / educational.

## What's next

[Notebook 12 — Capstone Agent](12_capstone_agent.ipynb): we wire everything together. User asks a voice question (multilingual, possibly code-switched), Whisper transcribes, ColPali retrieves the relevant page from a PDF index, GPT-4o produces a grounded answer, TTS speaks it back. The full S3 stack as one runnable agent.